<a href="https://colab.research.google.com/github/aisha13dikko-sudo/using-synthetic-data-for-thermal-comfort-classification/blob/main/wk15_terra_multirun_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# wk15: GPT-5.6 Terra, five traceable runs (corrected)

```
wk15_terra_multirun.ipynb  |  CORRECTED after reading the real notebook

This version reuses your actual build_spaced_windows, build_cold_targeted_
windows, build_temporal_prompt and run_temporal_experiment functions almost
unchanged, copied from temporal_llm_classification_clean.ipynb. The only
things added are: running Terra five times instead of once, logging every
run immediately, and never touching the GitHub token that lived in that
notebook's later cells (do not port cells 25-27 from the original; they are
not needed here and one of them contains a revoked but still-exposed token).

CORRECTIONS THIS VERSION MAKES TO WHAT I TOLD YOU BEFORE
  - Window construction is NOT plain systematic sampling. 50 windows per
    participant are drawn at random start positions (seeded, 42), and a
    SEPARATE function deliberately builds up to 20 more windows per
    participant that end specifically at a Cold-labelled row, searching
    backward from every Cold row in that participant's data. This exists
    because Cold is 0.6% of the built-in split and random sampling would
    almost never produce a Cold-ending window otherwise. LLM.tex describes
    this incorrectly as plain systematic sampling and needs correcting.
  - The total window count is NOT confirmed as 120. It depends on how much
    data participants 5 and 12 have. CELL 3 below prints the real count.
    Update LLM.tex with whatever this prints, not with 120.
  - Terra's API call includes no temperature parameter at all when
    temperature is None, rather than passing temperature=0. Passing 0 would
    likely have caused every call to fail or behave unexpectedly.

WHAT YOU STILL NEED TO DO
  Nothing pasted in by hand this time. Just run top to bottom. If any
  function signature here doesn't match your memory of the original,
  stop and check against temporal_llm_classification_clean.ipynb directly
  rather than assuming this transcription is perfect.
```


In [1]:
!pip install -q openai datasets scikit-learn 2>&1 | tail -2

import os, re, json, time, platform, warnings
from datetime import datetime

import numpy as np
import pandas as pd
import sklearn
from datasets import load_dataset
from openai import OpenAI
from sklearn.metrics import f1_score, classification_report

warnings.filterwarnings("ignore")
os.makedirs("results", exist_ok=True)

from google.colab import userdata
client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

MODEL = "gpt-5.6-terra"
N_RUNS = 5
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

MANIFEST = {
    "run_id": RUN_ID,
    "started": datetime.now().isoformat(timespec="seconds"),
    "python": platform.python_version(), "pandas": pd.__version__,
    "sklearn": sklearn.__version__, "model": MODEL, "n_runs": N_RUNS,
    "purpose": "five traceable GPT-5.6 Terra temporal runs, replacing four "
               "hand-typed values confirmed unrecoverable 16 Aug 2026",
}
RUN_RESULTS = []
PER_WINDOW = {}
print("Run ID:", RUN_ID)


Run ID: 20260816_150808


In [2]:
# Identical to the original notebook's data loading.
dataset = load_dataset("kopetri/AutoTherm", "indoor")
test_df = dataset["test"].to_pandas()

def extract_participant_id(filename):
    match = re.search(r"participant_\d+", filename)
    return match.group() if match else "unknown"

test_df["participant_id"] = test_df["file_name"].apply(extract_participant_id)
test_df = test_df.sort_values(["participant_id", "Timestamp"]).reset_index(drop=True)

LABEL_NAMES = {-3: "Cold", -2: "Cool", -1: "Slightly Cool", 0: "Neutral",
               1: "Slightly Warm", 2: "Warm", 3: "Hot"}

print(f"Built-in test split: {len(test_df):,} rows")
print("Participants:", sorted(test_df["participant_id"].unique()))


README.md:   0%|          | 0.00/8.57k [00:00<?, ?B/s]

indoor/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B / 29.8MB            

indoor/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

indoor/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B / 30.0MB            

indoor/train-00001-of-00002.parquet: downloading bytes:           |  0.00B            

indoor/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 7.41MB            

indoor/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1566728 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/194829 [00:00<?, ? examples/s]

Built-in test split: 194,829 rows
Participants: ['participant_12', 'participant_5']


In [3]:
# Window construction, copied from temporal_llm_classification_clean.ipynb
# cell 8, unchanged. Reusing this exactly is what keeps this re-run
# comparable to the original GPT-4o-mini temporal result (0.3033).

N_FRAMES = 10
FRAME_SPACING = 1260
TOTAL_SPAN = N_FRAMES * FRAME_SPACING
RANDOM_SEED = 42


def build_spaced_windows(p_df, n_samples=50, seed=RANDOM_SEED):
    windows = []
    max_start = len(p_df) - TOTAL_SPAN
    if max_start < 1:
        return windows
    np.random.seed(seed)
    start_positions = np.random.choice(max_start, size=min(n_samples, max_start), replace=False)
    for start in start_positions:
        indices = [start + i * FRAME_SPACING for i in range(N_FRAMES)]
        window = p_df.iloc[indices].copy()
        window["frame_number"] = range(1, N_FRAMES + 1)
        windows.append(window)
    return windows


def build_cold_targeted_windows(p_df, n_samples=20, seed=RANDOM_SEED):
    windows = []
    cold_indices = p_df[p_df["Label"] == -3].index.tolist()
    valid_cold = [i for i in cold_indices if i >= TOTAL_SPAN]
    if not valid_cold:
        return windows
    np.random.seed(seed)
    selected = np.random.choice(valid_cold, size=min(n_samples, len(valid_cold)), replace=False)
    for end_idx in selected:
        indices = [end_idx - (N_FRAMES - 1 - i) * FRAME_SPACING for i in range(N_FRAMES)]
        indices = [max(0, idx) for idx in indices]
        window = p_df.iloc[indices].copy()
        window["frame_number"] = range(1, N_FRAMES + 1)
        windows.append(window)
    return windows


regular_windows = []
cold_windows = []
for participant in sorted(test_df["participant_id"].unique()):
    p_df = test_df[test_df["participant_id"] == participant].reset_index(drop=True)
    regular_windows.extend(build_spaced_windows(p_df, n_samples=50))
    cold_windows.extend(build_cold_targeted_windows(p_df, n_samples=20))

all_windows = regular_windows + cold_windows

print(f"Regular windows:       {len(regular_windows)}")
print(f"Cold-targeted windows: {len(cold_windows)}")
print(f"Total windows:         {len(all_windows)}")
print()
print(">>> THIS IS THE REAL WINDOW COUNT. Use it in LLM.tex, not 120. <<<")

target_labels = [w.iloc[-1]["Label"] for w in all_windows]
print("\nTarget label distribution (frame 10):")
print(pd.Series(target_labels).value_counts().sort_index())


Regular windows:       100
Cold-targeted windows: 20
Total windows:         120

>>> THIS IS THE REAL WINDOW COUNT. Use it in LLM.tex, not 120. <<<

Target label distribution (frame 10):
-3    22
-2    14
-1    20
 0    25
 1    21
 2    16
 3     2
Name: count, dtype: int64


In [4]:
# Prompt construction, copied from cell 10, unchanged. This is the design
# that shows nine ground-truth labels before asking for the tenth, already
# disclosed as a limitation in LLM.tex Section 6.2. Preserved deliberately.

def build_temporal_prompt(window):
    context_rows = window.iloc[:-1]
    target_row   = window.iloc[-1]
    sequence = ""
    for i, (_, row) in enumerate(context_rows.iterrows()):
        label = int(row["Label"])
        time_ago = (N_FRAMES - 1 - i) * 30
        sequence += (
            f"Frame {i+1} ({time_ago}s ago): "
            f"Ambient={row['Ambient_Temperature']:.1f}\u00b0C, "
            f"Wrist={row['Wrist_Skin_Temperature']:.2f}\u00b0C, "
            f"Radiation={row['Radiation-Temp']:.1f}\u00b0C, "
            f"Humidity={row['Ambient_Humidity']:.0f}%, "
            f"GSR={row['GSR']:.3f}, "
            f"HR={row['Heart_Rate']:.1f}bpm "
            f"-> {label} ({LABEL_NAMES[label]})\n"
        )
    sequence += (
        f"Frame 10 (now): "
        f"Ambient={target_row['Ambient_Temperature']:.1f}\u00b0C, "
        f"Wrist={target_row['Wrist_Skin_Temperature']:.2f}\u00b0C, "
        f"Radiation={target_row['Radiation-Temp']:.1f}\u00b0C, "
        f"Humidity={target_row['Ambient_Humidity']:.0f}%, "
        f"GSR={target_row['GSR']:.3f}, "
        f"HR={target_row['Heart_Rate']:.1f}bpm "
        f"-> ?"
    )
    prompt = (
        "You are a thermal comfort expert analysing wearable sensor data "
        "from a person in an indoor office. Readings are spaced 30 seconds "
        "apart covering the last 5 minutes.\n\n"
        "Thermal comfort scale: "
        "-3=Cold, -2=Cool, -1=Slightly Cool, 0=Neutral, "
        "1=Slightly Warm, 2=Warm, 3=Hot\n\n"
        f"{sequence}\n\n"
        "Look carefully at the trend across all 10 frames. Consider whether "
        "temperatures are rising, falling, or stable. Use the full scale "
        "including extreme values like -3 or 3 if the trend suggests it.\n\n"
        "Respond with exactly one integer only: -3, -2, -1, 0, 1, 2, or 3."
    )
    return prompt, int(target_row["Label"])


In [5]:
# Experiment runner, copied from cell 12, unchanged. Note the temperature
# handling: nothing is passed for Terra, not temperature=0. Passing 0 would
# likely fail, since the model rejects any value other than 1.

def run_temporal_experiment(model_name, windows, use_system_prompt=False,
                            temperature=None, delay=0.3):
    predictions = []
    actuals = []
    system_message = (
        "You are a thermal comfort classification expert. "
        "You must respond with exactly one integer from -3 to 3. "
        "Never respond with anything else. "
        "Never default to 0 unless the data strongly suggests neutral comfort."
    )
    for i, window in enumerate(windows):
        try:
            prompt, actual_label = build_temporal_prompt(window)
            messages = []
            if use_system_prompt:
                messages.append({"role": "system", "content": system_message})
            messages.append({"role": "user", "content": prompt})

            call_kwargs = {"model": model_name, "messages": messages,
                           "max_completion_tokens": 10}
            if temperature is not None:
                call_kwargs["temperature"] = temperature

            response = client.chat.completions.create(**call_kwargs)
            prediction_text = response.choices[0].message.content.strip()
            try:
                prediction = int(prediction_text)
                if prediction not in range(-3, 4):
                    prediction = 0
            except ValueError:
                prediction = 0
        except Exception as e:
            print(f"  API error on window {i}: {e}")
            prediction = 0
            actual_label = int(window.iloc[-1]["Label"])

        predictions.append(prediction)
        actuals.append(actual_label)
        time.sleep(delay)
        if (i + 1) % 20 == 0:
            print(f"  {i+1}/{len(windows)}")
    return predictions, actuals


In [6]:
# The only genuinely new part: run Terra five times, log and save every run
# immediately rather than only at the end.

for run_idx in range(1, N_RUNS + 1):
    print(f"\n=== Terra run {run_idx}/{N_RUNS} ===")
    t0 = time.time()
    preds, actuals = run_temporal_experiment(
        model_name=MODEL, windows=all_windows,
        use_system_prompt=True, temperature=None,
    )
    elapsed = time.time() - t0

    macro_f1 = f1_score(actuals, preds, average="macro", zero_division=0)
    cold_f1 = f1_score(actuals, preds, labels=[-3], average="macro", zero_division=0)

    RUN_RESULTS.append({
        "run": run_idx, "macro_f1": round(float(macro_f1), 4),
        "cold_f1": round(float(cold_f1), 4), "n_windows": len(all_windows),
        "elapsed_s": round(elapsed, 1),
    })
    PER_WINDOW[f"run_{run_idx}"] = {"actuals": actuals, "predictions": preds}

    print(f"  Run {run_idx}: macro F1 = {macro_f1:.4f}, Cold F1 = {cold_f1:.4f}")

    # Save after every run. A dead runtime on run 4 must not cost runs 1-3.
    pd.DataFrame(RUN_RESULTS).to_csv("results/wk15_terra_runs.csv", index=False)
    json.dump(PER_WINDOW, open("results/wk15_terra_predictions.json", "w"), indent=2)



=== Terra run 1/5 ===
  20/120
  40/120
  60/120
  80/120
  100/120
  120/120
  Run 1: macro F1 = 0.1268, Cold F1 = 0.0000

=== Terra run 2/5 ===
  20/120
  40/120
  60/120
  80/120
  100/120
  120/120
  Run 2: macro F1 = 0.0682, Cold F1 = 0.0000

=== Terra run 3/5 ===
  20/120
  40/120
  60/120
  80/120
  100/120
  120/120
  Run 3: macro F1 = 0.0496, Cold F1 = 0.0000

=== Terra run 4/5 ===
  20/120
  40/120
  60/120
  API error on window 76: Error code: 400 - {'error': {'message': 'Could not finish the message because max_tokens or model output limit was reached. Please try again with higher max_tokens.', 'type': 'invalid_request_error', 'param': None, 'code': None}}
  80/120
  100/120
  120/120
  Run 4: macro F1 = 0.0671, Cold F1 = 0.0000

=== Terra run 5/5 ===
  20/120
  40/120
  60/120
  80/120
  100/120
  120/120
  Run 5: macro F1 = 0.0496, Cold F1 = 0.0000


In [7]:
res = pd.DataFrame(RUN_RESULTS)
print(res.to_string(index=False))

mean_f1 = res["macro_f1"].mean()
sd_f1 = res["macro_f1"].std()
print(f"\nMean macro F1 across {len(res)} runs: {mean_f1:.4f}")
print(f"SD: {sd_f1:.4f}")
print(f"Range: {res['macro_f1'].min():.4f} to {res['macro_f1'].max():.4f}")
print(f"\nWindow count used: {len(all_windows)} (confirm this matches Cell 3's output)")
print("\nRetired figures, do not use: mean 0.0866, SD 0.0653, built from one")
print("real run and four unverifiable ones. Not a target to match.")

MANIFEST["finished"] = datetime.now().isoformat(timespec="seconds")
MANIFEST["mean_macro_f1"] = round(float(mean_f1), 4)
MANIFEST["sd_macro_f1"] = round(float(sd_f1), 4)
MANIFEST["individual_runs"] = RUN_RESULTS
MANIFEST["n_windows"] = len(all_windows)
MANIFEST["n_regular_windows"] = len(regular_windows)
MANIFEST["n_cold_targeted_windows"] = len(cold_windows)

with open("results/wk15_manifest.json", "w") as f:
    json.dump(MANIFEST, f, indent=2)

from google.colab import files
files.download("results/wk15_terra_runs.csv")
files.download("results/wk15_terra_predictions.json")
files.download("results/wk15_manifest.json")
print("\nSaved and downloaded.")


 run  macro_f1  cold_f1  n_windows  elapsed_s
   1    0.1268      0.0        120      130.7
   2    0.0682      0.0        120      126.1
   3    0.0496      0.0        120      127.0
   4    0.0671      0.0        120      127.2
   5    0.0496      0.0        120      129.0

Mean macro F1 across 5 runs: 0.0723
SD: 0.0318
Range: 0.0496 to 0.1268

Window count used: 120 (confirm this matches Cell 3's output)

Retired figures, do not use: mean 0.0866, SD 0.0653, built from one
real run and four unverifiable ones. Not a target to match.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Saved and downloaded.
